# Часть 2 — Ретривер по базе знаний академии Lamoda

Гибридный поиск (sparse + dense) поверх статей, собранных парсером из Части 1.
Всё работает **в памяти ноутбука**, без внешних векторных БД и сервисов (Qdrant,
pgvector, FAISS-server и т.п.) — таково требование задания.

**Архитектура и обоснование (кратко):**

| Слой | Что | Зачем |
|---|---|---|
| Чанкинг | разбиение статьи по заголовкам H2/H3 + overlap | заголовок — естественная смысловая граница; чанк влезает в контекст и точно цитируется (url + раздел) |
| Sparse | **BM25** по лемматизированным токенам (`pymorphy3`) | ловит точные термины: коды, `FBO`, «штрихкод», имена полей API — там, где эмбеддинг «размывает» |
| Dense | **intfloat/multilingual-e5-base** (косинус) | ловит перефраз и синонимы; e5 силён на русском; префиксы `query:`/`passage:` обязательны |
| Fusion | **RRF** (Reciprocal Rank Fusion) | у BM25 и косинуса несравнимые шкалы; RRF работает по рангам — без подбора весов, устойчиво |
| Выдача | top-K чанков + ссылка на источник | критерий задания: результат с указанием источника |

Функция-интерфейс для ReAct-агента (Часть 3): `search_knowledge_base(query, k)`.


## 1. Установка зависимостей

`rank_bm25` — лексический поиск; `sentence-transformers` — эмбеддинги e5; `pymorphy3` — лемматизация русского (нормализуем словоформы перед BM25).

In [ ]:
%pip install -q rank_bm25 sentence-transformers pymorphy3
print("deps installed")

## 2. Загрузка статей

Читаем `data/articles.jsonl` — результат парсера (Часть 1). В едином ноутбуке этот файл уже создан выше; при отдельном запуске убедитесь, что он на месте (или загрузите его в Colab).

In [ ]:
import json
from pathlib import Path

ART_PATH = Path("data/articles.jsonl")
assert ART_PATH.exists(), (
    f"Не найден {ART_PATH}. Сначала запустите парсер (Часть 1), "
    "либо загрузите articles.jsonl в Colab."
)

ARTICLES = [json.loads(line) for line in ART_PATH.read_text(encoding="utf-8").splitlines() if line.strip()]
print(f"загружено статей: {len(ARTICLES)}")
print("пример полей:", list(ARTICLES[0].keys()))

## 3. Чанкинг

Статью целиком в индекс класть плохо: длинный текст «усредняет» эмбеддинг и
ухудшает попадание. Режем по заголовкам **H2/H3** (`## …`, `### …`) — это
авторские смысловые блоки. Если блок длиннее лимита, добиваем по абзацам со
**сдвигом-overlap**, чтобы не разрезать мысль на границе.

Каждый чанк несёт метаданные (`url`, `title`, `breadcrumbs`, `heading`) — без них
нельзя сослаться на источник в ответе агента.

In [ ]:
import re

MAX_CHARS = 1200
OVERLAP   = 200
MIN_CHARS = 80

_HEADING_RE = re.compile(r"(?m)^(#{2,3}\s+.+)$")

def split_sections(md_text):
    """Разбить markdown на (заголовок, тело) по H2/H3. Текст до первого
    заголовка идёт как вводная секция с пустым заголовком."""
    parts = _HEADING_RE.split(md_text)
    sections = []
    intro = parts[0].strip()
    if intro:
        sections.append(("", intro))
    for i in range(1, len(parts), 2):
        heading = parts[i].strip()
        body = parts[i + 1].strip() if i + 1 < len(parts) else ""
        sections.append((heading, body))
    return sections

def window_paragraphs(text, max_chars, overlap):
    """Если секция длиннее max_chars — нарезать по абзацам с перекрытием.
    Большие монолитные блоки (markdown-таблицы, длинные списки) разбиваем
    дополнительно по строкам, иначе e5 (лимит ~512 токенов) обрежет хвост."""
    if len(text) <= max_chars:
        return [text]
    atoms = []
    for p in text.split("\n\n"):
        p = p.strip()
        if not p:
            continue
        if len(p) <= max_chars:
            atoms.append(p)
        else:
            buf = ""
            for line in p.split("\n"):
                if buf and len(buf) + len(line) + 1 > max_chars:
                    atoms.append(buf)
                    buf = line
                else:
                    buf = (buf + "\n" + line) if buf else line
            if buf:
                atoms.append(buf)
    chunks, cur = [], ""
    for a in atoms:
        if cur and len(cur) + len(a) + 2 > max_chars:
            chunks.append(cur.strip())
            tail = cur[-overlap:]
            cur = (tail + "\n\n" + a)
        else:
            cur = (cur + "\n\n" + a) if cur else a
    if cur.strip():
        chunks.append(cur.strip())
    return chunks

CHUNKS = []
for art in ARTICLES:
    for heading, body in split_sections(art["markdown"]):
        section_text = (heading + "\n" + body).strip() if heading else body
        for piece in window_paragraphs(section_text, MAX_CHARS, OVERLAP):
            if len(piece) < MIN_CHARS:
                continue
            CHUNKS.append({
                "text": piece,
                "url": art["url"],
                "title": art["title"],
                "breadcrumbs": art.get("breadcrumbs", []),
                "heading": heading,
            })

print(f"статей: {len(ARTICLES)}  ->  чанков: {len(CHUNKS)}")
_lens = [len(c["text"]) for c in CHUNKS]
print(f"длина чанка (симв.): min={min(_lens)}  avg={sum(_lens)//len(_lens)}  max={max(_lens)}")

## 4. Sparse-индекс: BM25 + лемматизация

BM25 — классический лексический рейтинг (как в поисковых движках). Он силён там,
где эмбеддинги слабы: точные коды, аббревиатуры (`FBO`, `API`, `SKU`), редкие
имена полей. Перед индексацией:

- приводим к нижнему регистру, токенизируем по `\w+`;
- **лемматизируем** (`pymorphy3`): «штрихкоды» → «штрихкод», «поставки» →
  «поставка» — иначе словоформы не совпадут с запросом;
- выкидываем стоп-слова и однобуквенные токены.

In [ ]:
import pymorphy3
from rank_bm25 import BM25Okapi

morph = pymorphy3.MorphAnalyzer()
_TOKEN_RE = re.compile(r"\w+", re.UNICODE)

STOP = set("""и в во не что он на я с со как а то все она так его но да ты к у же
вы за бы по только ее мне было вот от меня еще нет о из ему теперь когда даже ну
ли если или быть был него до вас нибудь опять уж вам ведь там потом себя ничего
ей может они тут где есть надо ней для мы тебя их чем была сам чтоб без будто чего
раз тоже себе под будет ж тогда кто этот того потому этого какой совсем ним здесь
этом один почти мой тем чтобы нее были куда зачем всех никогда можно при наконец
два об другой хоть после над больше тот через эти нас про всего них какая много
разве сказать три эту моя впрочем хорошо свою этой перед иногда лучше чуть том
нельзя такой им более всегда конечно всю между это""".split())

def tokenize(text):
    toks = _TOKEN_RE.findall(text.lower())
    out = []
    for t in toks:
        if len(t) < 2 or t in STOP:
            continue
        lemma = morph.parse(t)[0].normal_form
        if lemma in STOP:
            continue
        out.append(lemma)
    return out

print("строим BM25-индекс (лемматизация всех чанков)...")
CORPUS_TOKENS = [tokenize(c["text"]) for c in CHUNKS]
bm25 = BM25Okapi(CORPUS_TOKENS)
print(f"BM25 готов: {len(CORPUS_TOKENS)} документов")
print("пример токенов:", CORPUS_TOKENS[0][:12])

## 5. Dense-индекс: эмбеддинги multilingual-e5-small

`intfloat/multilingual-e5-small` — многоязычная модель, сильная на русском.
Берём **small** (а не base): на нашем небольшом корпусе (~2.3 тыс. чанков)
качество почти не отличается, зато она **быстро считается на CPU** — ноутбук не
требует переключения среды на GPU.

Важная деталь e5: тексты надо помечать префиксами — **`passage: `** для
документов и **`query: `** для запроса, иначе качество заметно падает.

Эмбеддинги нормализуем (L2), тогда **косинусная близость = скалярное
произведение** — быстрый матричный поиск через numpy, без FAISS/внешней БД.
Матрицу эмбеддингов кешируем на диск: повторный запуск ноутбука не пересчитывает
её заново.

In [ ]:
import numpy as np
import hashlib
from sentence_transformers import SentenceTransformer

EMB_MODEL = "intfloat/multilingual-e5-small"
CACHE_DIR = Path("data"); CACHE_DIR.mkdir(exist_ok=True)

_sig = hashlib.sha1((EMB_MODEL + "||" + "".join(c["text"] for c in CHUNKS)).encode("utf-8")).hexdigest()[:16]
EMB_CACHE = CACHE_DIR / f"emb_{_sig}.npy"

print(f"загрузка модели {EMB_MODEL}...")
model = SentenceTransformer(EMB_MODEL)

if EMB_CACHE.exists():
    EMB = np.load(EMB_CACHE)
    print(f"эмбеддинги взяты из кеша: {EMB_CACHE.name}  shape={EMB.shape}")
else:
    passages = ["passage: " + c["text"] for c in CHUNKS]
    EMB = model.encode(passages, batch_size=64, normalize_embeddings=True,
                       show_progress_bar=True).astype(np.float32)
    np.save(EMB_CACHE, EMB)
    print(f"эмбеддинги посчитаны и сохранены: {EMB_CACHE.name}  shape={EMB.shape}")

## 6. Гибридный поиск: RRF-слияние

У BM25 (неограниченные баллы) и косинуса (`[-1, 1]`) **несравнимые шкалы** —
складывать их напрямую нельзя. Берём **Reciprocal Rank Fusion**: каждый документ
получает `1 / (RRF_K + rank)` от каждого ранкера и баллы суммируются. RRF смотрит
только на **порядок**, поэтому устойчив и не требует подбора весов
(`RRF_K = 60` — стандартное значение из статьи Cormack et al.).

`search_knowledge_base(query, k)` — единый интерфейс, который дальше вызывает
ReAct-агент (Часть 3): возвращает top-K чанков со ссылкой на источник.

In [ ]:
from collections import defaultdict

RRF_K = 60
CANDIDATES = 50

def search_knowledge_base(query, k=5):
    """Гибридный поиск по базе знаний академии Lamoda.
    Возвращает список из k чанков: text, title, url, breadcrumbs, heading, score."""
    q_tokens = tokenize(query)
    bm_scores = bm25.get_scores(q_tokens)
    bm_top = np.argsort(bm_scores)[::-1][:CANDIDATES]

    q_emb = model.encode("query: " + query, normalize_embeddings=True).astype(np.float32)
    cos = EMB @ q_emb
    dn_top = np.argsort(cos)[::-1][:CANDIDATES]

    rrf = defaultdict(float)
    for rank, idx in enumerate(bm_top):
        rrf[int(idx)] += 1.0 / (RRF_K + rank)
    for rank, idx in enumerate(dn_top):
        rrf[int(idx)] += 1.0 / (RRF_K + rank)

    ranked = sorted(rrf.keys(), key=lambda i: rrf[i], reverse=True)[:k]
    results = []
    for idx in ranked:
        c = CHUNKS[idx]
        results.append({
            "text": c["text"],
            "title": c["title"],
            "url": c["url"],
            "breadcrumbs": c["breadcrumbs"],
            "heading": c["heading"],
            "score": round(float(rrf[idx]), 5),
        })
    return results

def show_results(query, k=5):
    print(f"ЗАПРОС: {query}\n" + "=" * 70)
    for i, r in enumerate(search_knowledge_base(query, k), 1):
        path = " / ".join(r["breadcrumbs"][1:]) if r["breadcrumbs"] else r["title"]
        snippet = " ".join(r["text"].split())[:240]
        print(f"\n[{i}] score={r['score']}  {r['title']}")
        print(f"    путь: {path}")
        print(f"    url:  {r['url']}")
        print(f"    {snippet}...")

print("search_knowledge_base() готова")

## 7. Проверка на реальных запросах

Несколько запросов разного типа: точный термин (BM25-сильный), перефраз/смысл (dense-сильный), и общий вопрос.

In [ ]:
show_results("какой формат штрихкода на товарной этикетке FBO", k=3)

In [ ]:
show_results("как подключиться к API селлера", k=3)

In [ ]:
show_results("сертификат соответствия на товар", k=3)

## 8. Что можно улучшить (план)

- **Реранкер** (cross-encoder, напр. `bge-reranker`) поверх top-N RRF — заметно
  поднимает точность, но требует ещё одной модели; для теста оставлен RRF.
- **Взвешенный RRF / настройка весов** под тип запроса (короткий код vs длинный
  вопрос).
- **Семантический чанкинг** (по смыслу, а не по заголовкам) для статей без чёткой
  структуры H2/H3.
- **Кеш токенизации/лемматизации** запросов и индекса для ускорения.
- **Метаданные в скоринг**: бустить чанки из релевантного раздела (breadcrumbs)
  или свежие (`updated`).

# Часть 3 — ReAct-агент-консультант

Агент отвечает на вопросы продавцов, **сам решая**, когда обратиться к базе
знаний. Реализован цикл **ReAct** (Reasoning + Acting): модель чередует
рассуждение (*Thought*), вызов инструмента (*Action*) и получение результата
(*Observation*), пока не сформулирует *Final Answer* со ссылками на источники.

**Решения и обоснование:**

- **Провайдер — Groq** (бесплатный ключ без карты, OpenAI-совместимый API,
  модель Llama 3.3 70B — хорошо знает русский).
- **Цикл ReAct написан вручную**, без LangGraph/ADK (требование задания). Это
  также делает код **провайдеро-независимым**: меняется только обёртка `llm_call`.
- **Один инструмент** — `search_knowledge_base` из Части 2 (sparse+dense поиск).
- Агент **обязан** хотя бы раз сходить в базу знаний и **не выдумывать факты** —
  отвечает только по тексту из *Observation* и указывает URL-источники.


## 9. Подключение к Groq

Нужен бесплатный **API-ключ** с console.groq.com (*API Keys → Create API Key*,
ключ вида `gsk_...`, карта не нужна). API совместим с OpenAI, поэтому сообщения
передаём как есть (`role`/`content`).

**Гибкое получение ключа** — три источника по приоритету:
1. переменная окружения `GROQ_API_KEY` (или Colab Secrets);
2. ключ автора в `AUTHOR_GROQ_KEY` ниже (для запуска «из коробки»);
3. ручной ввод через защищённое поле (если первые два пусты).

Проверяющий может либо подставить **свой** ключ (env / поле ввода), либо
воспользоваться приложенным ключом автора.

In [ ]:
%pip install -q groq
import os
from getpass import getpass
from groq import Groq

AUTHOR_GROQ_KEY = "gsk_ВПИШИТЕ_СЮДА_КЛЮЧ_АВТОРА"

def _resolve_groq_key():
    env_key = os.environ.get("GROQ_API_KEY", "").strip()
    if env_key:
        return env_key
    if AUTHOR_GROQ_KEY.startswith("gsk_") and "ВПИШИТЕ" not in AUTHOR_GROQ_KEY:
        return AUTHOR_GROQ_KEY
    return getpass("Groq API key (gsk_...): ").strip()

GROQ_API_KEY = _resolve_groq_key()
client = Groq(api_key=GROQ_API_KEY)
GROQ_MODEL = "llama-3.3-70b-versatile"

def llm_call(messages, temperature=0.1):
    """Обёртка над Groq (OpenAI-совместимый API): список dict-сообщений -> текст."""
    resp = client.chat.completions.create(
        model=GROQ_MODEL,
        messages=messages,
        temperature=temperature,
    )
    return resp.choices[0].message.content

print(llm_call([{"role": "user", "content": "Ответь одним словом: связь есть?"}]))

## 10. ReAct-цикл

Системный промпт задаёт протокол: на каждом шаге модель выводит либо
`Action: search_knowledge_base: <запрос>`, либо `Final Answer: <ответ>`.
Парсим текст регулярками, при `Action` — реально вызываем ретривер и
возвращаем результат как `Observation`. Цикл ограничен `max_steps` (защита от
зацикливания).

In [ ]:
import re

SYSTEM_PROMPT = """Ты — консультант академии селлеров Lamoda. Помогаешь продавцам,
отвечая на их вопросы строго по базе знаний академии.

Доступный инструмент:
  search_knowledge_base — поиск релевантных фрагментов статей.
  Аргумент: поисковый запрос на русском языке.

Работай по протоколу ReAct. На каждом шаге выводи РОВНО один из двух вариантов.

Когда нужна информация:
Thought: <что именно нужно узнать>
Action: search_knowledge_base: <поисковый запрос>

Когда информации достаточно:
Thought: <короткий вывод>
Final Answer: <ответ продавцу. В конце перечисли источники — URL статей из наблюдений>

Жёсткие правила:
- Перед Final Answer сделай минимум один поиск (Action).
- Бери факты ТОЛЬКО из блоков Observation. Ничего не выдумывай.
- Если в базе нет ответа — честно скажи об этом.
- В Final Answer обязательно указывай URL источников."""

_ACTION_RE = re.compile(r"Action:\s*search_knowledge_base:\s*(.+)", re.IGNORECASE)
_FINAL_RE  = re.compile(r"Final Answer:\s*(.+)", re.IGNORECASE | re.DOTALL)

def format_observation(results):
    blocks = []
    for i, r in enumerate(results, 1):
        snippet = " ".join(r["text"].split())[:400]
        blocks.append(f"[{i}] {r['title']} ({r['url']})\n{snippet}")
    return "\n\n".join(blocks)

def run_agent(question, max_steps=5, k=5, verbose=True):
    """ReAct-цикл: Thought -> Action(search_knowledge_base) -> Observation -> ... -> Final Answer."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": question},
    ]
    for step in range(1, max_steps + 1):
        reply = llm_call(messages)
        if verbose:
            print(f"\n----- шаг {step} -----\n{reply}")

        final = _FINAL_RE.search(reply)
        if final:
            return final.group(1).strip()

        action = _ACTION_RE.search(reply)
        if action:
            query = action.group(1).strip().strip('"').strip()
            results = search_knowledge_base(query, k)
            if verbose:
                print(f"[tool] search_knowledge_base({query!r}) -> {len(results)} фрагментов")
            messages.append({"role": "assistant", "content": reply})
            messages.append({"role": "user", "content": "Observation:\n" + format_observation(results)})
        else:
            messages.append({"role": "assistant", "content": reply})
            messages.append({"role": "user", "content": "Используй формат 'Action: search_knowledge_base: ...' или 'Final Answer: ...'."})

    return "Не удалось сформировать финальный ответ за отведённое число шагов."

print("run_agent() готов")

## 11. Диалог с агентом (e2e)

Видно полный цикл: агент рассуждает, вызывает `search_knowledge_base`, получает фрагменты и выдаёт финальный ответ со ссылками.

In [ ]:
answer = run_agent("Какой формат штрихкода нужен для поставки товара на склад Lamoda по модели FBO?")
print("\n=========== ОТВЕТ АГЕНТА ===========\n")
print(answer)

In [ ]:
print(run_agent("Нужен ли сертификат соответствия на товар и какой документ оформить?", verbose=False))

## 12. Что можно улучшить у агента (план)

- **Нативный function-calling Groq** (`tools=[...]`) вместо текстового парсинга
  `Action:` — надёжнее, но сильнее завязан на конкретный провайдер; для
  наглядности цикла оставлен ручной ReAct.
- **Несколько инструментов** (поиск по разделу, уточняющий вопрос пользователю).
- **Память диалога** между вопросами (сейчас каждый вызов независим).
- **Guardrails**: пост-проверка, что все факты ответа подкреплены источниками.
- **Реранкер** в `search_knowledge_base` (см. план Части 2) — точнее Observation,
  меньше шагов до ответа.